#  Sloth's Slow-Motion Hotel – Main (CLI Test)
Konsolenbasierter Test der Hauptfunktionen. 

Testet die Buchungsvalidierung, die Movement-Tracker-Logik und das **State Pattern**.

## Importe

In [1]:
from pydantic import ValidationError
from models import HammockBooking, MovementTracker
from states import RestingState, SleepingState, EatingState

## State Pattern Demo
Das State Pattern steuert das Verhalten des Faultier-Gastes. 
Je nach Zustand (Ausruhen, Schlafen, Essen) reagiert der Gast unterschiedlich auf Aktionen.

In [2]:
state = RestingState()
print(f"--- State: {state.name} ---")
print(f"Eat:   {state.eat()}")
print(f"Sleep: {state.sleep()}")
print(f"Move:  {state.move()}")

--- State: Resting ---
Eat:   Slowly munching on a leaf... Delicious.
Sleep: Eyes closing... transitioning to sleep.
Move:  Moving very slowly to the hammock.


In [3]:
state = SleepingState()
print(f"--- State: {state.name} ---")
result = state.eat()
print(f"Eat:   {result}")
assert "Cannot" in result, "SleepingState.eat() should return an ERROR!"
print(f"Sleep: {state.sleep()}")
print(f"Move:  {state.move()}")

--- State: Sleeping ---
Eat:   Cannot eat while sleeping! Wake up first.
Sleep: Already asleep... Zzzzz...
Move:  Dreaming of moving... but staying put.


In [4]:
state = EatingState()
print(f"--- State: {state.name} ---")
print(f"Eat:   {state.eat()}")
print(f"Sleep: {state.sleep()}")
print(f"Move:  {state.move()}")

--- State: Eating ---
Eat:   Already eating. Don't rush me.
Sleep: Too full to sleep yet.
Move:  Cannot move while chewing. Safety first.


In [5]:
print(">>> State Transition Demo")

current_state = RestingState()
print(f"Start:       {current_state.name}")

# Resting -> eat -> EatingState
result = current_state.eat()
assert "ERROR" not in result
current_state = EatingState()
print(f"-> eat()  -> {current_state.name}")

# Eating -> sleep -> SleepingState
result = current_state.sleep()
current_state = SleepingState()
print(f"-> sleep()-> {current_state.name}")

# Sleeping -> wake up -> RestingState
current_state = RestingState()
print(f"-> wake  -> {current_state.name}")

print("State Pattern test passed ")

>>> State Transition Demo
Start:       Resting
-> eat()  -> Eating
-> sleep()-> Sleeping
-> wake  -> Resting
State Pattern test passed 


## Movement Tracker Tests
Der Rabatt wird basierend auf der Schrittzahl berechnet. 
Weniger Schritte = höherer Rabatt (typisch Faultier!).

In [6]:
print("--- Test Case A: Very Lazy Sloth (50 Steps) ---")
tracker_lazy = MovementTracker(steps_today=50)
discount_lazy = tracker_lazy.calculate_discount()
print(f"Guest 'Sid' walked {tracker_lazy.steps_today} steps.")
print(f"Discount granted: {discount_lazy * 100}%")
assert discount_lazy == 0.50, "Lazy sloth should get 50% discount!"
print("Result: Excellent! Barely moved. Gold Tier Laziness!")

--- Test Case A: Very Lazy Sloth (50 Steps) ---
Guest 'Sid' walked 50 steps.
Discount granted: 50.0%
Result: Excellent! Barely moved. Gold Tier Laziness!


In [7]:
print("--- Test Case B: Moderate Sloth (250 Steps) ---")
tracker_moderate = MovementTracker(steps_today=250)
discount_moderate = tracker_moderate.calculate_discount()
print(f"Guest 'Manny' walked {tracker_moderate.steps_today} steps.")
print(f"Discount granted: {discount_moderate * 100}%")
assert discount_moderate == 0.20, "Moderate sloth should get 20% discount!"
print("Result: Silver Tier – could be lazier.")

--- Test Case B: Moderate Sloth (250 Steps) ---
Guest 'Manny' walked 250 steps.
Discount granted: 20.0%
Result: Silver Tier – could be lazier.


In [8]:
print("--- Test Case C: Active Sloth (600 Steps) ---")
tracker_active = MovementTracker(steps_today=600)
discount_active = tracker_active.calculate_discount()
print(f"Guest 'Flash' walked {tracker_active.steps_today} steps.")
print(f"Discount granted: {discount_active * 100}%")
assert discount_active == 0.00, "Active sloth should get 0% discount!"
print("Result: Too active! No discount – this is a sloth hotel!")

print("Movement Tracker test passed ")

--- Test Case C: Active Sloth (600 Steps) ---
Guest 'Flash' walked 600 steps.
Discount granted: 0.0%
Result: Too active! No discount – this is a sloth hotel!
Movement Tracker test passed 


## Duck Typing (Universeller Gast)
Dieser Abschnitt demonstriert, wie Sid das Python Duck Typing nutzt. 

Anstatt zu prüfen, ob ein Gast von einer Basisklasse `HotelGuest` erbt, interessiert Sid nur, ob der Gast die Methode `get_slowness_factor()` implementiert. Schauen wir uns an, wie verschiedene Entitäten wie Schildkröten, Pandas und überarbeitete IU Dozenten beim Check-in abschneiden!

In [9]:
from models import Sloth, Turtle, Panda, IUDozent, check_in_guest

print("--- Hotel Check-In Desk ---")

guests = [
    ("Frank the Sloth", Sloth()),
    ("Krickel the Turtle", Turtle()),
    ("Alexander the Panda", Panda()),
    ("Sebastian the IU Dozent", IUDozent()),
    ("A standard integer (no slowness factor)", 42),  # Intentionally missing method
]

for name, entity in guests:
    print(f"Checking in {name}:")
    response = check_in_guest(entity)
    print(f"> Sid says: {response}")

--- Hotel Check-In Desk ---
Checking in Frank the Sloth:
> Sid says: Welcome! Your slowness factor is 1.0. Here is your hammock.
Checking in Krickel the Turtle:
> Sid says: Welcome! Your slowness factor is 0.8. Here is your hammock.
Checking in Alexander the Panda:
> Sid says: Welcome! Your slowness factor is 0.6. Here is your hammock.
Checking in Sebastian the IU Dozent:
> Sid says: Welcome! Your slowness factor is 0.9. Here is your hammock.
Checking in A standard integer (no slowness factor):
> Sid says: Security! This entity doesn't know how to be slow!


### Duck Typing im Movement Tracker
Der `MovementTracker` berechnet Rabatte nun dynamisch anhand des `slowness_factor` des übergebenen Gastes. Schauen wir uns an, wie viel Rabatt die verschiedene Spezies für exakt dieselbe Anzahl an Schritten erhalten!

In [10]:
from models import MovementTracker

print("--- Discount Calculation based on Species ---")
for name, entity in guests:
    if isinstance(entity, int):  # Skip the integer used to test error handling earlier
        continue
    tracker = MovementTracker(steps_today=50, guest=entity)
    discount = tracker.calculate_discount()
    print(f"{name} walked 50 steps -> Gets {discount * 100:.1f}% discount!")

--- Discount Calculation based on Species ---
Frank the Sloth walked 50 steps -> Gets 50.0% discount!
Krickel the Turtle walked 50 steps -> Gets 40.0% discount!
Alexander the Panda walked 50 steps -> Gets 30.0% discount!
Sebastian the IU Dozent walked 50 steps -> Gets 45.0% discount!


## Hängemattenbuchungs-Tests
Buchungen werden mit Pydantic validiert. 
Der Mindestaufenthalt beträgt 7 Nächte (in CONFIG konfiguriert).

In [11]:
print("--- Test Case D: Valid Booking (10 Nights) ---")
try:
    booking = HammockBooking(guest_name="Sid", nights=10)
    print(
        f"SUCCESS: Booking for {booking.guest_name} confirmed for {booking.nights} nights."
    )
except ValidationError as e:
    print(f"ERROR: {e}")

--- Test Case D: Valid Booking (10 Nights) ---
SUCCESS: Booking for Sid confirmed for 10 nights.


In [12]:
print("--- Test Case E: Minimum Booking (7 Nights) ---")
try:
    booking_min = HammockBooking(guest_name="Diego", nights=7)
    print(
        f"SUCCESS: Booking for {booking_min.guest_name} confirmed for {booking_min.nights} nights."
    )
    print("Result: Exact minimum stay accepted.")
except ValidationError as e:
    print(f"ERROR: {e}")

--- Test Case E: Minimum Booking (7 Nights) ---
SUCCESS: Booking for Diego confirmed for 7 nights.
Result: Exact minimum stay accepted.


In [13]:
print("--- Test Case F: Invalid Booking (3 Nights – Too Short) ---")
try:
    print("Attempting to book for 3 nights...")
    HammockBooking(guest_name="Flash", nights=3)
    print("SUCCESS: Booking confirmed.")
except ValidationError as e:
    print("BLOCKED: System correctly rejected the booking.")
    print(f"Reason: {e.errors()[0]['msg']}")

print("Hammock Booking test passed ")

--- Test Case F: Invalid Booking (3 Nights – Too Short) ---
Attempting to book for 3 nights...
BLOCKED: System correctly rejected the booking.
Reason: Value error, Too stressful! Min 7 nights required. (REQ-FR-01)
Hammock Booking test passed 


## 3. Reifegrad-Rechner Tests (REQ-FR-05)
Dieser Abschnitt verifiziert die Klasse `MaturityCalculator`, welche **REQ-FR-05** implementiert. 
Nahrung wird nur freigegeben, wenn `days_on_branch` $\ge$ `optimal_maturity_days` ist.

In [14]:
from models import MaturityCalculator

print("--- Maturity Calculator Tests ---")

unripe_leaf = MaturityCalculator(
    item_name="Eucalyptus", days_on_branch=10, optimal_maturity_days=14
)
ripe_leaf = MaturityCalculator(
    item_name="Eucalyptus", days_on_branch=15, optimal_maturity_days=14
)

print(f"Leaf 1 (10 days): Ripe? {unripe_leaf.is_ripe()}")
print(f"Leaf 2 (15 days): Ripe? {ripe_leaf.is_ripe()}")

--- Maturity Calculator Tests ---
Leaf 1 (10 days): Ripe? False
Leaf 2 (15 days): Ripe? True


## Wissenschaftliche Herausforderung: Zyklomatische Komplexität (State Pattern vs. Anti-Pattern)
Dieser Abschnitt untersucht die Forschungshypothese: 
**"Wie verändert sich die Codequalität, gemessen durch das Radon-Tool, wenn das State Pattern in der Architektur verwendet wird?"** 

Wir vergleichen die implementierte `states.py` (die das State Pattern verwendet) mit einer neu erstellten `states_anti_pattern.py` (die eine einzelne Zustandsvariable und mehrere `if/elif`-Zweige verwendet).

In [15]:
import subprocess
subprocess.run(["pip", "install", "radon"], check=True)

CompletedProcess(args=['pip', 'install', 'radon'], returncode=0)

In [16]:
# Analyze the State Pattern implementation (states.py)
print("--- RADON ANALYSIS: STATE PATTERN (states.py) ---")
subprocess.run(["radon", "cc", "-s", "states.py"])

--- RADON ANALYSIS: STATE PATTERN (states.py) ---


CompletedProcess(args=['radon', 'cc', '-s', 'states.py'], returncode=0)

In [17]:
# Analyze the Anti-Pattern implementation (states_anti_pattern.py)
print("--- RADON ANALYSIS: ANTI-PATTERN (states_anti_pattern.py) ---")
subprocess.run(["radon", "cc", "-s", "states_anti_pattern.py"])

--- RADON ANALYSIS: ANTI-PATTERN (states_anti_pattern.py) ---


CompletedProcess(args=['radon', 'cc', '-s', 'states_anti_pattern.py'], returncode=0)

### Schlussfolgerung
Wie zu sehen ist, führt das **Anti-Pattern** zu wesentlich höheren Werten der Zyklomatischen Komplexität (CC) für seine Methoden (z. B. `eat()`, `sleep()`, `move()`), da jede Methode die Variable `self.state` mit `if/elif`-Blöcken überprüfen muss. 

Im Gegensatz dazu erreicht das **State Pattern** für fast alle Methoden einen CC-Wert von **1 (Note A)**, da das Verhalten in eigenständige Klassen (`RestingState`, `SleepingState`, `EatingState`) dezentralisiert ist. Der Code ist stark zusammenhängend, einfacher zu testen und bestätigt die Forschungshypothese: Das State Pattern reduziert die Komplexität auf Methodenebene drastisch.